In [3]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/data
!unzip -q /content/drive/MyDrive/archive.zip -d /content/data

Mounted at /content/drive


In [4]:
!rm -rf /content/data
!mkdir -p /content/data
!cp /content/drive/MyDrive/archive.zip /content/archive.zip
!unzip -qo /content/archive.zip -d /content/data
!ls /content/data

kaggle_3m  lgg-mri-segmentation


In [12]:
"""
U-Net (ResNet18 encoder) for the Kaggle LGG brain MRI dataset.

- Patient-grouped train/val/test split (no leakage).
- Trains a segmentation model on the masks, then turns it into a
  slice-level "tumor / no tumor" detector so it is comparable to the SVM.
- Runs on Kaggle (auto-detects data path + GPU) or locally.

Outputs (in /kaggle/working or current folder):
  unet_resnet18.pt  - best weights (by validation loss)
  results.json      - test metrics
"""
import os, glob, json, random, time,sys
import numpy as np
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from sklearn.model_selection import GroupShuffleSplit

# ------------------------------ config ------------------------------
IMG = 128           
EPOCHS = 25
BS = 32
LR = 3e-4
SEED = 42
QUICK_TEST = False   # True = 15 patients, 2 epochs: sanity check on laptop CPU
LOCAL_PATH = r"C:\Users\THOMAS\Downloads\archive\kaggle_3m"
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
else:
    sys.exit(1)
# ------------------------------ data ------------------------------
def find_base():
    for root in ["/kaggle/input", "/content/data"]:      # Kaggle, or Colab (unzip data to /content/data)
        hits = glob.glob(f"{root}/**/kaggle_3m", recursive=True)
        if hits:
            return min(hits, key=len)                    # shortest = avoids duplicate nested copy
    return LOCAL_PATH

base = find_base()
print("data folder:", base)
masks = sorted(glob.glob(os.path.join(base, "**", "*_mask.tif"), recursive=True))
pairs = [(m.replace("_mask.tif", ".tif"), m) for m in masks]
patients = np.array([os.path.basename(os.path.dirname(m)) for m in masks])
print(f"{len(pairs)} pairs, {len(set(patients))} patients (expect 3929 / 110)")

if QUICK_TEST:
    keep = np.isin(patients, sorted(set(patients))[:15])
    pairs = [p for p, k in zip(pairs, keep) if k]
    patients = patients[keep]
    EPOCHS = 2

def load(pairs):
    X = np.zeros((len(pairs), IMG, IMG, 3), np.uint8)
    Y = np.zeros((len(pairs), IMG, IMG), np.uint8)
    lab = np.zeros(len(pairs), np.int64)
    for i, (ip, mp) in enumerate(pairs):
        m = Image.open(mp).convert("L")
        lab[i] = int(np.asarray(m).max() > 0)            # label from the ORIGINAL mask
        X[i] = np.asarray(Image.open(ip).convert("RGB").resize((IMG, IMG), Image.BILINEAR))
        Y[i] = np.asarray(m.resize((IMG, IMG), Image.NEAREST)) > 0
        if i % 1000 == 0:
            print("loading", i, "/", len(pairs))
    return X, Y, lab

X, Y, lab = load(pairs)

# patient-grouped split: ~70 / 15 / 15
idx = np.arange(len(pairs))
trval, te = next(GroupShuffleSplit(1, test_size=0.15, random_state=SEED).split(idx, groups=patients))
tr, va = next(GroupShuffleSplit(1, test_size=0.15 / 0.85, random_state=SEED).split(trval, groups=patients[trval]))
tr, va = trval[tr], trval[va]
P = lambda ix: set(patients[ix])
assert not (P(tr) & P(va)) and not (P(tr) & P(te)) and not (P(va) & P(te)), "patient leakage!"
print(f"patients  train/val/test: {len(P(tr))}/{len(P(va))}/{len(P(te))}")
print(f"slices    train/val/test: {len(tr)}/{len(va)}/{len(te)}")

def to_dev(ix):
    x = torch.from_numpy(X[ix]).permute(0, 3, 1, 2).contiguous().to(device)   # uint8
    y = torch.from_numpy(Y[ix]).unsqueeze(1).to(device)                       # uint8 0/1
    return x, y

Xtr, Ytr = to_dev(tr); Xva, Yva = to_dev(va); Xte, Yte = to_dev(te)

# ------------------------------ model ------------------------------
def block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True),
        nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))

class Up(nn.Module):
    def __init__(self, i, skip, o):
        super().__init__()
        self.c = block(i + skip, o)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return self.c(torch.cat([x, skip], 1))

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        try:
            r = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
            print("loaded ImageNet-pretrained ResNet18 encoder")
        except Exception as e:
            print("pretrained weights unavailable (internet off?) -> training encoder from scratch:", e)
            r = torchvision.models.resnet18(weights=None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.u4 = Up(512, 256, 256); self.u3 = Up(256, 128, 128)
        self.u2 = Up(128, 64, 64);   self.u1 = Up(64, 64, 32)
        self.head = nn.Conv2d(32, 1, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1) * 255)
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1) * 255)

    def forward(self, x):                      # x: float, range 0-255
        x = (x - self.mean) / self.std
        x0 = self.stem(x)
        x1 = self.l1(self.pool(x0)); x2 = self.l2(x1); x3 = self.l3(x2); x4 = self.l4(x3)
        d = self.u4(x4, x3); d = self.u3(d, x2); d = self.u2(d, x1); d = self.u1(d, x0)
        d = F.interpolate(d, size=x.shape[2:], mode="bilinear", align_corners=False)
        return self.head(d)

def loss_fn(logits, y):                        # BCE + soft Dice
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1)
    return F.binary_cross_entropy_with_logits(logits, y) + dice.mean()

@torch.no_grad()
def predict_probs(model, Xd):
    model.eval()
    return torch.cat([torch.sigmoid(model(Xd[i:i + 64].float())).cpu() for i in range(0, len(Xd), 64)])

@torch.no_grad()
def val_loss(model):
    model.eval()
    tot = 0
    for i in range(0, len(Xva), 64):
        tot += loss_fn(model(Xva[i:i + 64].float()), Yva[i:i + 64].float()).item() * len(Xva[i:i + 64])
    return tot / len(Xva)

# ------------------------------ train ------------------------------
model = UNet().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
best, best_state = 1e9, None
n = len(Xtr)
for ep in range(EPOCHS):
    t0 = time.time(); model.train(); perm = torch.randperm(n, device=device); run = 0
    for i in range(0, n, BS):
        b = perm[i:i + BS]
        x, y = Xtr[b].float(), Ytr[b].float()
        f = torch.rand(len(b), device=device) < 0.5          # random horizontal flip
        x[f], y[f] = x[f].flip(-1), y[f].flip(-1)
        x = x * (0.9 + 0.2 * torch.rand(len(b), 1, 1, 1, device=device))   # brightness jitter
        loss = loss_fn(model(x.clamp(0, 255)), y)
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.item() * len(b)
    sched.step()
    vl = val_loss(model)
    if vl < best:
        best = vl
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, os.path.join(OUT, "unet_resnet18.pt"))
    print(f"epoch {ep + 1:02d}/{EPOCHS}  train {run / n:.4f}  val {vl:.4f}  best {best:.4f}  ({time.time() - t0:.0f}s)")

model.load_state_dict(best_state)

# ------------------------------ evaluate ------------------------------
# Slice decision: "tumor" if the predicted mask has >= MIN_PIXELS pixels.
# MIN_PIXELS is chosen on VALIDATION data only, then frozen for the test set.
def areas_of(probs):
    return (probs > 0.5).sum((1, 2, 3)).numpy()

def conf(pred, truth):
    tp = int(((pred == 1) & (truth == 1)).sum()); tn = int(((pred == 0) & (truth == 0)).sum())
    fp = int(((pred == 1) & (truth == 0)).sum()); fn = int(((pred == 0) & (truth == 1)).sum())
    return tp, tn, fp, fn

pv, pt = predict_probs(model, Xva), predict_probs(model, Xte)
av, at = areas_of(pv), areas_of(pt)
lv, lt = lab[va], lab[te]

best_thr, best_ba = 1, -1
for thr in [1, 5, 10, 20, 40, 80, 160]:
    tp, tn, fp, fn = conf((av >= thr).astype(int), lv)
    ba = 0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))     # balanced accuracy
    if ba > best_ba:
        best_ba, best_thr = ba, thr
print(f"\nchosen MIN_PIXELS (from val): {best_thr}  (val balanced acc {best_ba:.3f})")

pred = (at >= best_thr).astype(int)
tp, tn, fp, fn = conf(pred, lt)
acc = (tp + tn) / len(lt)
recall = tp / max(tp + fn, 1)
spec = tn / max(tn + fp, 1)
prec = tp / max(tp + fp, 1)

# pixel-level Dice on tumor slices only
pm = (pt > 0.5).float(); gt = Yte.cpu().float()
pos = torch.from_numpy(lt == 1)
inter = (pm[pos] * gt[pos]).sum((1, 2, 3))
dice = ((2 * inter + 1e-6) / (pm[pos].sum((1, 2, 3)) + gt[pos].sum((1, 2, 3)) + 1e-6)).mean().item()

print("\n=========== TEST (unseen patients) ===========")
print(f"accuracy     : {acc * 100:.2f}%")
print(f"tumor recall : {recall * 100:.2f}%")
print(f"specificity  : {spec * 100:.2f}%")
print(f"precision    : {prec * 100:.2f}%")
print(f"mean Dice on tumor slices: {dice:.3f}")
print(f"TP {tp}  TN {tn}  FP {fp}  FN {fn}")
print("SVM baseline (patient-grouped CV): accuracy 82.85%, recall 70.79%")

# recall by true tumor size (terciles of tumor area, test set)
true_area = Yte.cpu().sum((1, 2, 3)).numpy()
pa = true_area[lt == 1]
size_recall = {}
if len(pa) >= 3:
    q = np.quantile(pa, [1 / 3, 2 / 3])
    print("\nRecall by tumor size (true mask pixels):")
    for name, lo, hi in [("small ", 0, q[0]), ("medium", q[0], q[1]), ("large ", q[1], 1e9)]:
        sel = (lt == 1) & (true_area > lo if name != "small " else true_area >= 0) & (true_area <= hi)
        if sel.sum():
            r = float((pred[sel] == 1).mean())
            size_recall[name.strip()] = r
            print(f"  {name}: {r * 100:5.1f}%  ({int(sel.sum())} slices)")

json.dump({
    "img_size": IMG, "epochs": EPOCHS, "min_pixels": int(best_thr),
    "accuracy": acc, "recall": recall, "specificity": spec, "precision": prec,
    "dice_tumor_slices": dice, "confusion": {"tp": tp, "tn": tn, "fp": fp, "fn": fn},
    "recall_by_size": size_recall,
    "test_patients": sorted(P(te)),
}, open(os.path.join(OUT, "results.json"), "w"), indent=2)
print("\nsaved: unet_resnet18.pt, results.json in", OUT)


data folder: /content/data/kaggle_3m
3929 pairs, 110 patients (expect 3929 / 110)
loading 0 / 3929
loading 1000 / 3929
loading 2000 / 3929
loading 3000 / 3929
patients  train/val/test: 76/17/17
slices    train/val/test: 2795/601/533
loaded ImageNet-pretrained ResNet18 encoder
epoch 01/25  train 1.4891  val 1.3879  best 1.3879  (8s)
epoch 02/25  train 1.3393  val 1.2840  best 1.2840  (8s)
epoch 03/25  train 1.2378  val 1.2018  best 1.2018  (8s)
epoch 04/25  train 1.1491  val 1.1317  best 1.1317  (8s)
epoch 05/25  train 1.0720  val 1.0288  best 1.0288  (8s)
epoch 06/25  train 1.0057  val 0.9790  best 0.9790  (8s)
epoch 07/25  train 0.9545  val 0.9218  best 0.9218  (8s)
epoch 08/25  train 0.9093  val 0.9017  best 0.9017  (9s)
epoch 09/25  train 0.8737  val 0.8648  best 0.8648  (9s)
epoch 10/25  train 0.8483  val 0.8394  best 0.8394  (9s)
epoch 11/25  train 0.8238  val 0.8286  best 0.8286  (8s)
epoch 12/25  train 0.8076  val 0.8082  best 0.8082  (8s)
epoch 13/25  train 0.7944  val 0.7975  

In [11]:
print(open('/content/results.json').read())

{
  "img_size": 224,
  "epochs": 25,
  "min_pixels": 20,
  "accuracy": 0.924953095684803,
  "recall": 0.9540816326530612,
  "specificity": 0.9080118694362018,
  "precision": 0.8577981651376146,
  "dice_tumor_slices": 0.7963473796844482,
  "confusion": {
    "tp": 187,
    "tn": 306,
    "fp": 31,
    "fn": 9
  },
  "recall_by_size": {
    "small": 0.8636363636363636,
    "medium": 1.0,
    "large": 1.0
  },
  "test_patients": [
    "TCGA_CS_4941_19960909",
    "TCGA_CS_5393_19990606",
    "TCGA_CS_6290_20000917",
    "TCGA_CS_6665_20010817",
    "TCGA_DU_5852_19950709",
    "TCGA_DU_6407_19860514",
    "TCGA_DU_7306_19930512",
    "TCGA_DU_A5TS_19970726",
    "TCGA_FG_6688_20020215",
    "TCGA_FG_6692_20020606",
    "TCGA_FG_7637_20000922",
    "TCGA_HT_7602_19951103",
    "TCGA_HT_7605_19950916",
    "TCGA_HT_7608_19940304",
    "TCGA_HT_7616_19940813",
    "TCGA_HT_7686_19950629",
    "TCGA_HT_7882_19970125"
  ]
}


In [13]:
!cp /content/unet_resnet18.pt /content/results.json /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive | grep -E "unet_resnet18|results"

-rw------- 1 root root  931 Oct  4 07:32 results.json
-rw------- 1 root root  55M Oct  4 07:32 unet_resnet18.pt


In [ ]:
print("ho")

In [ ]:
import torch
m = model.eval().cpu()
traced = torch.jit.trace(m, torch.rand(1, 3, 224, 224) * 255)
traced.save("unet_traced.pt")
!cp unet_traced.pt /content/drive/MyDrive/
print("done")